# 04 - Time Series Analysis and Analytical Report

This notebook completes the Module 5 exercise: develop a comprehensive analytical report combining T-SQL data extraction, Python statistical analysis, and professional visualization.

You will:

1. Build a daily time series from financial indicators.
2. Decompose the series into trend, seasonal, and residual components.
3. Create a simple forecast.
4. Generate report-ready markdown output.
5. Evaluate the end-to-end pipeline against basic accuracy and performance benchmarks.
6. Optionally log the report run back to SQL Server.

In [ ]:
# Install required libraries for a fresh notebook environment such as Google Colab.
import sys
import subprocess

for package in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "sqlalchemy", "pyodbc", "python-dotenv"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"Package install skipped or failed for {package}: {exc}")

In [ ]:
import os
import time
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

try:
    import pyodbc
except Exception as exc:
    pyodbc = None
    print("pyodbc unavailable; SQL Server extraction/logging will be skipped:", exc)

try:
    from dotenv import load_dotenv
    from sqlalchemy import create_engine, text
except Exception as exc:
    load_dotenv = None
    create_engine = None
    text = None
    print("SQLAlchemy/dotenv unavailable; SQL Server extraction/logging will be skipped:", exc)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ENV_FILE = ".env"
sns.set_theme(style="whitegrid")
pipeline_start = time.perf_counter()

In [ ]:
# Shared data loader. It tries SQL Server first, then uses generated fallback data.
def build_sample_indicator_data():
    institutions = [
        ("CBL", "Central Bank Liquidity Desk", "Maseru", "Central Bank", 920_000_000, 310_000_000, 0.345, 0.218, 0.020, 1),
        ("MCB", "Maseru Commercial Bank", "Maseru", "Commercial Bank", 610_000_000, 520_000_000, 0.278, 0.161, 0.048, 2),
        ("LMB", "Leribe Microfinance Bank", "Leribe", "Microfinance", 185_000_000, 171_000_000, 0.235, 0.139, 0.073, 3),
        ("QFB", "Quthing Finance Bank", "Quthing", "Commercial Bank", 275_000_000, 251_000_000, 0.242, 0.148, 0.067, 4),
        ("BDB", "Butha-Buthe Development Bank", "Butha-Buthe", "Development Bank", 330_000_000, 298_000_000, 0.255, 0.154, 0.061, 5),
        ("MFI", "Mafeteng Inclusion Finance", "Mafeteng", "Microfinance", 145_000_000, 139_000_000, 0.218, 0.128, 0.086, 6),
    ]
    rows = []
    observation_id = 1
    for day_number, observation_date in enumerate(pd.date_range("2026-04-01", periods=60, freq="D")):
        for code, name, region, inst_type, deposits, loans, liquidity, capital, npl, weight in institutions:
            liquidity_ratio = round(liquidity + ((day_number % 10) * 0.0021) - (0.014 if weight in (3, 6) else 0) - (0.018 if 33 <= day_number <= 42 else 0), 4)
            capital_ratio = round(capital + ((day_number % 8) * 0.0014) - (0.011 if weight == 6 else 0), 4)
            npl_ratio = round(npl + ((day_number % 9) * 0.0018) + (0.010 if 33 <= day_number <= 42 else 0), 4)
            credit_growth = round(0.0310 + ((day_number % 12) * 0.0016) - (npl * 0.0800), 4)
            rows.append({
                "ObservationID": observation_id,
                "ObservationDate": observation_date,
                "InstitutionCode": code,
                "InstitutionName": name,
                "Region": region,
                "InstitutionType": inst_type,
                "LiquidityRatio": liquidity_ratio,
                "CapitalAdequacyRatio": capital_ratio,
                "NplRatio": npl_ratio,
                "TransactionVolume": 850 + (weight * 95) + (day_number * 8) + ((day_number % 6) * 35),
                "TransactionValueLSL": (deposits * 0.028) + (day_number * 185_000) + (weight * 75_000) + ((day_number % 7) * 430_000),
                "CreditGrowthRate": credit_growth,
                "InflationRate": round(0.0470 + ((day_number % 11) * 0.0007), 4),
                "InterbankRate": round(0.0820 + ((day_number % 7) * 0.0009), 4),
                "StressFlag": int(liquidity_ratio < 0.2300 or capital_ratio < 0.1300 or npl_ratio >= 0.0850 or credit_growth < 0.0280),
            })
            observation_id += 1
    return pd.DataFrame(rows)

def get_engine():
    if pyodbc is None or load_dotenv is None or create_engine is None:
        raise RuntimeError("SQL Server packages are unavailable")
    load_dotenv(BASE_DIR / ENV_FILE)
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "TrainingDB")
    user = os.getenv("DB_USER", "sa")
    password = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
    trusted = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")
    parts = [f"DRIVER={{{driver}}}", f"SERVER={server}", f"DATABASE={database}", "Encrypt=yes", "TrustServerCertificate=yes", "Connection Timeout=5"]
    parts.append("Trusted_Connection=yes" if trusted else f"UID={user};PWD={password}")
    return create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(';'.join(parts) + ';')}")

def load_indicator_data():
    query = """
    SELECT ObservationID, ObservationDate, InstitutionCode, InstitutionName, Region, InstitutionType,
           LiquidityRatio, CapitalAdequacyRatio, NplRatio, TransactionVolume, TransactionValueLSL,
           CreditGrowthRate, InflationRate, InterbankRate, CAST(StressFlag AS INT) AS StressFlag
    FROM m5.DailyFinancialIndicators
    ORDER BY ObservationDate, InstitutionCode;
    """
    try:
        engine = get_engine()
        data = pd.read_sql(query, engine, parse_dates=["ObservationDate"])
        source = "SQL Server"
    except Exception as exc:
        print("Using generated sample data. SQL Server unavailable:", exc)
        data = build_sample_indicator_data()
        source = "generated fallback data"
    print("Data source:", source)
    return data

df = load_indicator_data()
display(df.head())

## 1. Build the Daily Time Series

A time series is data ordered by time. Here we aggregate institution-level rows into one daily view.

In [ ]:
daily = (
    df.groupby("ObservationDate", as_index=False)
    .agg(
        LiquidityRatio=("LiquidityRatio", "mean"),
        NplRatio=("NplRatio", "mean"),
        TransactionValueLSL=("TransactionValueLSL", "sum"),
        StressRate=("StressFlag", "mean"),
    )
    .sort_values("ObservationDate")
)

daily["LiquidityRolling7"] = daily["LiquidityRatio"].rolling(7, min_periods=1).mean()
daily["DayIndex"] = np.arange(len(daily))
display(daily.head())
display(daily.tail())

## 2. Time Series Decomposition

Decomposition separates a time series into:

- Trend: long-term direction.
- Seasonal: repeating pattern.
- Residual: what remains after trend and seasonality are removed.

We use a weekly period of `7` because the sample data is daily.

In [ ]:
series = daily.set_index("ObservationDate")["LiquidityRatio"].asfreq("D")

# Simple additive decomposition for teaching:
# Observed = Trend + Seasonal + Residual
# Trend: 7-day centered rolling average.
# Seasonal: average weekday effect after removing trend.
# Residual: remaining movement not explained by trend or weekday pattern.
trend = series.rolling(window=7, center=True, min_periods=1).mean()
detrended = series - trend
weekday_effect = detrended.groupby(detrended.index.dayofweek).transform("mean")
seasonal = pd.Series(weekday_effect.to_numpy(), index=series.index, name="Seasonal")
residual = series - trend - seasonal

components = pd.DataFrame({
    "Observed": series,
    "Trend": trend,
    "Seasonal": seasonal,
    "Residual": residual,
})
components.to_csv(OUTPUT_DIR / "liquidity_decomposition_components.csv")

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
components["Observed"].plot(ax=axes[0], title="Observed Liquidity Ratio", color="#0F766E")
components["Trend"].plot(ax=axes[1], title="Trend Component", color="#2563EB")
components["Seasonal"].plot(ax=axes[2], title="Seasonal Component", color="#7C3AED")
components["Residual"].plot(ax=axes[3], title="Residual Component", color="#DC2626")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("Ratio")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "liquidity_decomposition.png", dpi=160)
plt.show()

display(components.head(10))

## 3. Simple Forecast

For beginner practice, we use linear regression on time index to forecast seven future days. This is intentionally simple, so students can focus on the workflow before learning advanced forecasting models.

In [ ]:
trend_model = LinearRegression()
trend_model.fit(daily[["DayIndex"]], daily["LiquidityRatio"])

future_days = pd.DataFrame({"DayIndex": np.arange(len(daily), len(daily) + 7)})
last_date = daily["ObservationDate"].max()
future_days["ObservationDate"] = pd.date_range(last_date + pd.Timedelta(days=1), periods=7)
future_days["ForecastLiquidityRatio"] = trend_model.predict(future_days[["DayIndex"]])
future_days.to_csv(OUTPUT_DIR / "liquidity_forecast.csv", index=False)
display(future_days)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=daily, x="ObservationDate", y="LiquidityRatio", ax=ax, label="Observed", color="#0F766E")
sns.lineplot(data=daily, x="ObservationDate", y="LiquidityRolling7", ax=ax, label="7-day rolling average", color="#7C3AED")
sns.lineplot(data=future_days, x="ObservationDate", y="ForecastLiquidityRatio", ax=ax, label="7-day forecast", color="#DC2626", linestyle="--")
ax.set_title("Liquidity Ratio Trend and Forecast")
ax.set_xlabel("Observation date")
ax.set_ylabel("Liquidity ratio")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "liquidity_forecast.png", dpi=160)
plt.show()

print("Interpretation reminder: the dashed forecast is a simple training forecast, not a production risk model.")

## 4. Analytical Report and Phase 1 Pipeline Benchmarks

The final report should combine SQL extraction context, Python statistical outputs, charts, model metrics, and time-series findings.

Benchmark checks used here:

- Data completeness: expected observation rows are present.
- Model reliability: model accuracy from notebook 03 meets a basic threshold.
- Visual outputs: required images exist.
- Pipeline runtime: notebook completes within a reasonable training threshold.

In [ ]:
metrics_path = OUTPUT_DIR / "model_metrics.csv"
model_accuracy = None
if metrics_path.exists():
    model_accuracy = float(pd.read_csv(metrics_path).loc[0, "accuracy"])

observation_rows = int(len(df))
mean_liquidity = float(df["LiquidityRatio"].mean())
mean_npl = float(df["NplRatio"].mean())
stress_rate = float(df["StressFlag"].mean())
pipeline_seconds = time.perf_counter() - pipeline_start

required_outputs = [
    "liquidity_trend.png",
    "npl_distribution_by_type.png",
    "correlation_heatmap.png",
    "stress_classifier_confusion_matrix.png",
    "liquidity_decomposition.png",
    "liquidity_forecast.png",
]
output_status = {name: (OUTPUT_DIR / name).exists() for name in required_outputs}

benchmarks = pd.DataFrame([
    {"benchmark": "Observation rows >= 300", "passed": observation_rows >= 300, "actual": observation_rows},
    {"benchmark": "Model accuracy >= 0.75", "passed": (model_accuracy is not None and model_accuracy >= 0.75), "actual": model_accuracy},
    {"benchmark": "All required visuals generated", "passed": all(output_status.values()), "actual": str(output_status)},
    {"benchmark": "Pipeline runtime <= 120 seconds", "passed": pipeline_seconds <= 120, "actual": round(pipeline_seconds, 2)},
])
benchmarks.to_csv(OUTPUT_DIR / "phase1_pipeline_benchmarks.csv", index=False)
display(benchmarks)

report_text = f"""# Module 5 Analytical Report Summary

## Data Extraction Context

- Source: `m5.DailyFinancialIndicators` when SQL Server is available; generated fallback data otherwise.
- Observation rows: {observation_rows}

## Statistical Findings

- Mean liquidity ratio: {mean_liquidity:.4f}
- Mean NPL ratio: {mean_npl:.4f}
- Stress observation rate: {stress_rate:.2%}

## Model Output

- Model accuracy: {model_accuracy if model_accuracy is not None else 'Run notebook 03 first'}

## Time Series Output

- Decomposition chart: `outputs/liquidity_decomposition.png`
- Forecast chart: `outputs/liquidity_forecast.png`

## Required Visuals

- `outputs/liquidity_trend.png`
- `outputs/npl_distribution_by_type.png`
- `outputs/correlation_heatmap.png`
- `outputs/stress_classifier_confusion_matrix.png`
- `outputs/liquidity_decomposition.png`
- `outputs/liquidity_forecast.png`

## Phase 1 Simulation Benchmark Summary

```text
{benchmarks.to_string(index=False)}
```
"""

(OUTPUT_DIR / "module5_report_summary.md").write_text(report_text, encoding="utf-8")
print(report_text)

## 5. Optional SQL Report Log

If SQL Server is available, this cell logs the analytical report run to `m5.AnalysisReportRun`. In Colab or offline mode, it skips safely.

In [ ]:
try:
    engine = get_engine()
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO m5.AnalysisReportRun
                (AnalystName, ObservationRows, MeanLiquidityRatio, MeanNplRatio, ModelAccuracy, Notes)
            VALUES
                (:analyst, :rows, :mean_liquidity, :mean_npl, :model_accuracy, :notes);
        """), {
            "analyst": "Training Participant",
            "rows": observation_rows,
            "mean_liquidity": round(mean_liquidity, 4),
            "mean_npl": round(mean_npl, 4),
            "model_accuracy": None if model_accuracy is None else round(model_accuracy, 4),
            "notes": "Module 5 report generated from notebook with benchmark checks.",
        })
    print("Report run logged to SQL Server.")
except Exception as exc:
    print("SQL report logging skipped:", exc)